# 11 — Sentiment e posizione verso ICE (articoli e commenti NYT)

Misuriamo **come** il NYT e i suoi lettori parlano di ICE, con due misure distinte:

- **Tono** (negativo / neutro / positivo): quanto è carico emotivamente il testo. Su un tema come ICE il tono dipende molto dall'argomento (una morte in detenzione è negativa comunque la si racconti), quindi da solo non dice da che parte sta il testo.
- **Posizione verso ICE** (critica / neutra / favorevole): il giudizio sull'agenzia espresso dal testo, separato da quello delle fonti che cita. È la misura che servirà per il confronto con Telegram.

1. gold set di 152 articoli annotato a mano, con la posizione riletta alla cieca (§1.2);
2. modelli classici validati sul gold set e **scartati**: VADER e RoBERTa per il tono, NLI zero-shot per la posizione (§1.3);
3. LLM (Claude) con le linee guida di annotazione come prompt, messo a punto su 52 articoli del gold set (§1.4).

**Input**
- `data/processed/nyt/articles_ice.parquet` (2.093 articoli ICE).
- `data/processed/nyt/comments_ice_pseudo.parquet` (commenti pseudonimizzati dal notebook 09 §1: nessun `user_id` né nome).
- `data/annotations/nyt_sentiment_articles_gold_v2.csv` (gold set rivisto, input umano).

## 0. Setup

In [ ]:
import json
import logging
import re
from pathlib import Path

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import torch
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.metrics import cohen_kappa_score, confusion_matrix, f1_score
from transformers import pipeline
from transformers.utils import logging as transformers_logging

%matplotlib inline

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = BASE_DIR / "data" / "processed" / "nyt"
ANNOTATIONS_DIR = BASE_DIR / "data" / "annotations"
PLOTS_DIR = PROCESSED_DIR / "plots"

IN_ARTICLES = PROCESSED_DIR / "articles_ice.parquet"
IN_COMMENTS = PROCESSED_DIR / "comments_ice_pseudo.parquet"

# gold set annotati a mano: input umano, mai rigenerati né sovrascritti dal notebook
GOLD_ARTICLES = ANNOTATIONS_DIR / "nyt_sentiment_articles_gold_v2.csv"
GOLD_COMMENTS = ANNOTATIONS_DIR / "nyt_sentiment_comments_gold.csv"

OUT_ARTICLE_SENTIMENT = PROCESSED_DIR / "articles_ice_sentiment.parquet"
OUT_COMMENT_SENTIMENT = PROCESSED_DIR / "comments_ice_sentiment.parquet"

ROBERTA_MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"
SEED = 42

for d in (ANNOTATIONS_DIR, PLOTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", force=True)
log = logging.getLogger("nyt_sentiment")
# le richieste HTTP di huggingface_hub (una riga per file scaricato) coprirebbero il resto del log
for noisy in ("httpx", "huggingface_hub"):
    logging.getLogger(noisy).setLevel(logging.WARNING)
transformers_logging.set_verbosity_error()

DEVICE = 0 if torch.cuda.is_available() else -1
torch.manual_seed(SEED)
log.info("torch %s, device: %s, thread CPU: %d", torch.__version__, "cuda" if DEVICE == 0 else "cpu", torch.get_num_threads())

# Stessa palette del notebook 09: un colore per periodo, in ordine fisso
PERIOD_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
MAIN_COLOR = PERIOD_COLORS[0]
MUTED_COLOR = "#b8b8b0"
# tono / posizione: negativo (o critico) = rosso, neutro = grigio, positivo (o favorevole) = blu
LABEL_COLORS = {"negative": "#d64545", "neutral": MUTED_COLOR, "positive": "#2a78d6"}

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e6e6e1",
    "grid.linewidth": 0.6,
    "axes.axisbelow": True,
})


def get_periods(max_year: int) -> list[tuple[str, int, int]]:
    """Stessi periodi dei notebook 02-10: (label, anno iniziale incluso, anno finale escluso)."""
    return [
        ("2003-2016", 2003, 2017),
        ("2017-2020", 2017, 2021),
        ("2021-2024", 2021, 2025),
        ("2025-presente", 2025, max_year + 1),
    ]


def assign_period(years: pd.Series) -> pd.Series:
    periods = get_periods(int(years.max()))
    labels = [label for label, _, _ in periods]
    bins = [periods[0][1]] + [end for _, _, end in periods]
    return pd.cut(years, bins=bins, right=False, labels=labels)

### 0.1 Modelli di riferimento

**VADER** (lessico + regole, Hutto & Gilbert 2014) e **RoBERTa** `cardiffnlp/twitter-roberta-base-sentiment-latest` (3 classi) per il tono. Sono risultati troppo deboli (§1.3) e restano solo come termine di confronto per l'LLM. Il tokenizer di RoBERTa serve anche a controllare la lunghezza dei testi (§1.1).

In [ ]:
nltk.download("vader_lexicon", quiet=True)
vader = SentimentIntensityAnalyzer()
roberta = pipeline("sentiment-analysis", model=ROBERTA_MODEL, tokenizer=ROBERTA_MODEL, device=DEVICE, truncation=True, max_length=512)

## 1. Articoli NYT

### 1.1 Testo da classificare

Si lavora su `semantic_text` = **titolo + abstract + lead**, costruito come nel notebook 06:
- il **dateline** iniziale del lead ("WASHINGTON —") è tolto: non porta tono né posizione;
- quando abstract e lead coincidono (succede spesso) il lead non viene ripetuto, altrimenti lo stesso testo conterebbe due volte.

In [ ]:
# Dateline NYT: "CITTÀ[, STATO] —" in maiuscolo a inizio lead (stessa regex del notebook 06)
DATELINE_RE = re.compile(r"^[A-Z][A-Z .,'’-]{2,40}?\s*[—–]\s*")


def load_articles() -> pd.DataFrame:
    df = pd.read_parquet(IN_ARTICLES, columns=["id", "pub_date", "headline", "abstract", "lead_paragraph", "section_name", "web_url", "word_count"])
    for col in ["headline", "abstract", "lead_paragraph"]:
        df[col] = df[col].fillna("").astype(str)

    lead = df["lead_paragraph"].str.replace(DATELINE_RE, "", n=1, regex=True)
    lead = lead.where(df["abstract"].str.strip() != df["lead_paragraph"].str.strip(), "")
    df["semantic_text"] = (
        (df["headline"] + ". " + df["abstract"] + " " + lead)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip(" .")
    )
    df["year"] = df["pub_date"].dt.year
    df["period"] = assign_period(df["year"])
    return df


articles = load_articles()
empty = articles["semantic_text"].str.len() == 0
if empty.any():
    log.warning("%d articoli con semantic_text vuoto", empty.sum())

articles["n_words"] = articles["semantic_text"].str.split().str.len()
articles["n_tokens"] = [len(ids) for ids in roberta.tokenizer(articles["semantic_text"].tolist())["input_ids"]]
log.info("Articoli: %d, token RoBERTa: mediana %d, massimo %d (limite 512), sopra il limite: %d",
         len(articles), articles["n_tokens"].median(), articles["n_tokens"].max(), (articles["n_tokens"] > 512).sum())
articles.groupby("period", observed=True).size().rename("articoli").to_frame().T

### 1.2 Gold set

152 articoli ICE (**38 per periodo**, estratti a caso dentro ogni periodo, seed 42) annotati a mano il 2026-10-07 con due etichette, **tono** (`neg`/`neu`/`pos`) e **posizione verso ICE** (`crit`/`neu`/`fav`), secondo linee guida scritte prima dell'annotazione (testo completo in `docs/sentiment_analysis.md`; il prompt della §1.4 ne è la traduzione in inglese). La regola centrale: le critiche riportate da fonti terze ("gli avvocati dicono che...") sono posizione **neutra**, conta il giudizio del giornale. L'etichetta è data sul `semantic_text`, cioè sul testo che leggono i modelli.

**Gold set rivisto** (`nyt_sentiment_articles_gold_v2.csv`, l'originale non è toccato). Le posizioni sono state rilette alla cieca il 2026-10-15: 117 articoli riletti, 46 dei 91 critici diventati neutri. La prima annotazione aveva spesso seguito il tono (80 degli 88 negativi erano anche critici, ora 42). Distribuzione finale: 99 neutri, 45 critici, 8 favorevoli; il tono è quello della prima annotazione. Accordo dell'annotatore con sé stesso sugli articoli riletti: kappa 0,26, misurato soprattutto sui casi a rischio: la posizione è difficile anche per una persona, e un kappa modesto dei modelli va letto tenendone conto.

**Pesi.** Un campione casuale sull'intero corpus sarebbe ~60% 2025+ con pochissimi articoli 2021-2024; con lo stesso numero per periodo l'accuratezza si stima con la stessa precisione ovunque. Per riportare le metriche al corpus intero si usa `sample_weight` = articoli del periodo / articoli annotati del periodo (un articolo 2025+ "vale" molti più articoli di uno 2021-2024).

In [ ]:
# forme brevi ed estese delle etichette nel gold set
TONE_KEYS = {"-": "neg", "neg": "neg", "0": "neu", "neu": "neu", "+": "pos", "pos": "pos"}
STANCE_KEYS = {"c": "crit", "crit": "crit", "0": "neu", "neu": "neu", "f": "fav", "fav": "fav"}
GOLD_TONE = {key: {"neg": "negative", "neu": "neutral", "pos": "positive"}[v] for key, v in TONE_KEYS.items()}
STANCE_LABELS = ["crit", "neu", "fav"]

gold = pd.read_csv(GOLD_ARTICLES, dtype=str, keep_default_na=False)
gold["gold_tone"] = gold["tone"].str.lower().map(GOLD_TONE)
gold["gold_stance"] = gold["stance"].str.lower().map(STANCE_KEYS)
assert gold[["gold_tone", "gold_stance"]].notna().all().all(), "etichette non riconosciute nel gold set"

labels = articles["period"].cat.categories.tolist()
period_sizes = articles["period"].astype(str).value_counts()
gold["sample_weight"] = gold["period"].map(period_sizes) / gold["period"].map(gold["period"].value_counts())
log.info("Gold set: %d articoli; tono %s; posizione %s", len(gold),
         gold["gold_tone"].value_counts().to_dict(), gold["gold_stance"].value_counts().to_dict())

### 1.3 Modelli classici: perché non li usiamo

Prima dell'LLM sono stati validati sul gold set i modelli classici: VADER e RoBERTa per il tono (grafici sotto), NLI zero-shot per la posizione (solo numeri, in fondo alla sezione).

**Da punteggio a etichetta**: VADER con le soglie standard (`compound ≥ 0,05` positivo, `≤ −0,05` negativo, in mezzo neutro, non regolate sul gold set per non adattarle al campione); RoBERTa con la classe più probabile.

**Metriche**: **accuratezza** (quota di etichette uguali a quelle manuali); **F1 macro** (media dell'F1 delle tre classi, penalizza chi ignora una classe rara come `positive`, 7 articoli); **kappa di Cohen** (accordo oltre il caso: 0 = a caso, 1 = perfetto); **riferimento minimo** (*Majority baseline*: risponde sempre con la classe più frequente, il minimo da battere). Intervalli al 95% da un **bootstrap** di 1.000 ricampionamenti dentro ogni periodo; metriche sul corpus intero pesate con `sample_weight`, per periodo non pesate.

In [ ]:
TONE_LABELS = ["negative", "neutral", "positive"]
VADER_THRESHOLD = 0.05


def score_tone(texts: list[str], batch_size: int = 16) -> pd.DataFrame:
    # Punteggi grezzi di entrambi i modelli: la stessa funzione serve poi sul corpus intero
    compound = [vader.polarity_scores(t)["compound"] for t in texts]
    probs = roberta(texts, top_k=None, batch_size=batch_size)
    scores = pd.DataFrame([{f"roberta_p_{d['label']}": d["score"] for d in p} for p in probs])
    scores.insert(0, "vader_compound", compound)
    scores["vader_tone"] = np.select(
        [scores["vader_compound"] >= VADER_THRESHOLD, scores["vader_compound"] <= -VADER_THRESHOLD],
        ["positive", "negative"], "neutral")
    scores["roberta_tone"] = scores[[f"roberta_p_{l}" for l in TONE_LABELS]].to_numpy().argmax(axis=1)
    scores["roberta_tone"] = scores["roberta_tone"].map(dict(enumerate(TONE_LABELS)))
    return scores


gold_eval = pd.concat([gold.reset_index(drop=True), score_tone(gold["semantic_text"].tolist())], axis=1)
gold_eval["baseline_tone"] = gold_eval["gold_tone"].mode()[0]

log.info("Gold set valutato: %d articoli; tono manuale %s", len(gold_eval), gold_eval["gold_tone"].value_counts().to_dict())

In [ ]:
TONE_MODELS = {"vader_tone": "VADER", "roberta_tone": "RoBERTa", "baseline_tone": "Majority baseline"}
N_BOOT = 1000


def classification_metrics(y_true: pd.Series, y_pred: pd.Series, labels: list[str], weight: pd.Series | None = None) -> dict[str, float]:
    return {
        "accuracy": np.average(y_true == y_pred, weights=weight),
        "macro_f1": f1_score(y_true, y_pred, labels=labels, average="macro", sample_weight=weight, zero_division=0),
        "kappa": cohen_kappa_score(y_true, y_pred, labels=labels, sample_weight=weight),
    }


def bootstrap_metrics(df: pd.DataFrame, gold_col: str, pred_col: str, labels: list[str], weighted: bool, n_boot: int = N_BOOT) -> pd.DataFrame:
    # ricampionamento stratificato per periodo: ogni replica conserva 38 articoli per periodo come il gold set
    rng = np.random.default_rng(SEED)
    groups = [g.index.to_numpy() for _, g in df.groupby("period")]
    reps = []
    for _ in range(n_boot):
        idx = np.concatenate([rng.choice(g, size=len(g), replace=True) for g in groups])
        s = df.loc[idx]
        reps.append(classification_metrics(s[gold_col], s[pred_col], labels, s["sample_weight"] if weighted else None))
    return pd.DataFrame(reps)


def validation_scores(df: pd.DataFrame, gold_col: str, models: dict[str, str], labels: list[str]) -> pd.DataFrame:
    """Metriche con intervallo bootstrap al 95%: corpus intero (pesato) e per periodo (non pesato)."""
    rows = []
    for col, name in models.items():
        subsets = [("All (weighted)", df, True)] + [(p, g, False) for p, g in df.groupby("period")]
        for period, g, weighted in subsets:
            point = classification_metrics(g[gold_col], g[col], labels, g["sample_weight"] if weighted else None)
            boot = bootstrap_metrics(g, gold_col, col, labels, weighted)
            for metric, value in point.items():
                lo, hi = boot[metric].quantile([0.025, 0.975])
                rows.append({"model": name, "period": period, "metric": metric, "value": value, "lo": lo, "hi": hi})
    return pd.DataFrame(rows)


tone_scores = validation_scores(gold_eval, "gold_tone", TONE_MODELS, TONE_LABELS)

In [ ]:
period_order = ["All (weighted)"] + labels
metric_titles = {"accuracy": "Accuracy", "macro_f1": "Macro F1", "kappa": "Cohen's kappa"}


def plot_validation(scores: pd.DataFrame, model_colors: dict[str, str], title: str, filename: str) -> None:
    n = len(model_colors)
    fig, axes = plt.subplots(1, 3, figsize=(12, 2.4 + 0.45 * n), sharey=True)
    for ax, (metric, metric_title) in zip(axes, metric_titles.items()):
        for k, (name, color) in enumerate(model_colors.items()):
            s = scores[(scores["model"] == name) & (scores["metric"] == metric)].set_index("period").loc[period_order]
            y = np.arange(len(period_order)) + (k - (n - 1) / 2) * 0.7 / n
            ax.errorbar(s["value"], y, xerr=[s["value"] - s["lo"], s["hi"] - s["value"]], fmt="o", color=color,
                        ms=5, capsize=2, lw=1.2, label=name)
        ax.set_title(metric_title)
        ax.set_xlim(-0.1 if metric == "kappa" else 0, 1)
        ax.axhline(0.5, color="#1f1f1f", lw=0.6)
    axes[0].set_yticks(range(len(period_order)), period_order)
    axes[0].invert_yaxis()
    fig.legend(*axes[0].get_legend_handles_labels(), loc="lower center", ncol=n, fontsize=8, frameon=False, bbox_to_anchor=(0.5, -0.06))
    fig.suptitle(title, y=1.0)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / filename, dpi=150, bbox_inches="tight")
    plt.show()


plot_validation(tone_scores, {"VADER": PERIOD_COLORS[1], "RoBERTa": PERIOD_COLORS[0], "Majority baseline": MUTED_COLOR},
                "Tone: model vs manual labels on the gold set (95% bootstrap intervals)", "sentiment_tone_validation.png")

La prima riga è il corpus intero (pesato), sotto la linea i quattro periodi (38 articoli ciascuno, non pesati). Due modelli sono distinguibili solo se gli intervalli si sovrappongono poco; un modello è stabile nel tempo se i suoi punti restano allineati tra i periodi.

In [ ]:
gold_eval["roberta_polarity"] = gold_eval["roberta_p_positive"] - gold_eval["roberta_p_negative"]
rng = np.random.default_rng(SEED)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
for ax, (col, title, thr) in zip(axes, [("vader_compound", "VADER compound", VADER_THRESHOLD),
                                        ("roberta_polarity", "RoBERTa P(positive) − P(negative)", None)]):
    for i, label in enumerate(TONE_LABELS):
        x = gold_eval.loc[gold_eval["gold_tone"] == label, col]
        ax.scatter(x, i + rng.uniform(-0.18, 0.18, len(x)), s=14, alpha=0.7, color=LABEL_COLORS[label])
        ax.plot([x.median()] * 2, [i - 0.3, i + 0.3], color="#1f1f1f", lw=1.5)
    if thr:
        for t in (-thr, thr):
            ax.axvline(t, color="#1f1f1f", lw=0.6, ls="--")
    ax.set_xlim(-1.05, 1.05)
    ax.set_xlabel(title)
axes[0].set_yticks(range(3), [f"{l} (n={(gold_eval['gold_tone'] == l).sum()})" for l in TONE_LABELS])
axes[0].set_ylabel("Manual label")
axes[0].invert_yaxis()
fig.suptitle("Model scores by manual tone label")
fig.tight_layout()
fig.savefig(PLOTS_DIR / "sentiment_tone_scores.png", dpi=150)
plt.show()

un punto per articolo del gold set, raggruppati per etichetta manuale; il trattino nero è la mediana. Asse x: a sinistra il punteggio continuo di VADER (linee tratteggiate = soglie ±0,05), a destra la differenza tra probabilità positiva e negativa di RoBERTa. Se un modello separa bene le classi, le tre nuvole stanno in zone diverse dell'asse; se si sovrappongono, nessuna soglia le può distinguere.

#### Perché non abbiamo scelto questi modelli

**VADER e RoBERTa per il tono**
- **Accordo troppo basso.** RoBERTa, il migliore, ha accuratezza 0,62, F1 macro 0,42 e kappa 0,28 (pesati). In accuratezza supera appena il riferimento minimo; VADER (kappa 0,13) è poco sopra il caso.
- **Non distinguono neutro e negativo**, ed è il caso più frequente. Le nuvole `negative` e `neutral` si sovrappongono su quasi tutto l'asse (secondo grafico), quindi nessuna soglia diversa le separerebbe. Parole come *detention*, *deportation* e *arrest* sono negative per il lessico anche nella cronaca neutra. I 7 positivi non vengono riconosciuti.
- **Instabili tra periodi.** RoBERTa ha kappa 0,53 nel 2003-2016 e circa 0 nel 2017-2020, quindi un andamento nel tempo del tono misurato con RoBERTa non sarebbe credibile.

**NLI zero-shot per la posizione** (`MoritzLaurer/deberta-v3-base-zeroshot-v2.0`, tre formulazioni dell'ipotesi; sezione tolta dal notebook il 2026-10-16)
- La formulazione principale ("The author of this text is critical of / neutral toward / supportive of ICE") ha kappa 0,27 sulla prima annotazione. L'accuratezza non supera il riferimento minimo, e le altre formulazioni danno lo stesso quadro.
- **Peggiora nel tempo**, fino a kappa 0,06 nel 2025+ (58% del corpus). La cronaca dei raid e delle proteste descrive eventi conflittuali anche quando non giudica ICE, e l'NLI la legge come critica.
- Sul gold set rivisto va ancora peggio (kappa da 0,37 a 0,20, non pesato): con la prima annotazione era premiato quando seguiva il tono.

Nessuno dei tre sa separare il giudizio del giornale da quello delle fonti o dal carico emotivo dei fatti. Per questo nella §1.4 si prova un LLM che applica le stesse linee guida dell'annotatore. 

### 1.4 Tono e posizione con un LLM sul gold set

. Qui un LLM fa da secondo annotatore: riceve le linee guida di annotazione (§1.2) come prompt e restituisce tono e posizione.

**Impostazione.**
- **Prompt**: le linee guida in inglese senza nessun esempio.
- **Input**: sezione + `semantic_text`.
- **Output**: JSON con schema fisso (structured outputs dell'API): `evidence` (le parole del testo che decidono la posizione, oppure "none"), `tone` (neg/neu/pos), `stance` (crit/neu/fav), `ice_central` (ICE centrale o solo sfondo; non validato). Il riassunto del ragionamento interno del modello è salvato nel raw (`display: "summarized"`).
- **Modelli**: Claude Haiku 5.5, Sonnet 5.5 e Opus 5.5, tutti con effort `low` (il compito è una classificazione). Per Sonnet e Opus c'è un **fallback**: se il modello rifiuta, l'API ripete la richiesta su un altro modello, e quale ha risposto resta in `served_model`. Haiku non ha fallback.
- **Raw**: ogni risposta completa dell'API è salvata in `data/raw/llm/nyt_sentiment/<fase>/<modello>_run<n>.jsonl` prima del parsing. Un articolo già presente nel file non viene richiesto di nuovo (checkpoint), quindi rieseguire le celle non ripete chiamate né costi.
- **Costo**: prezzi per milione di token in `LLM_PRICES`; la spesa è calcolata dai token restituiti dall'API.

In [ ]:
import os
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone

import anthropic
from dotenv import load_dotenv
from sklearn.exceptions import UndefinedMetricWarning

LLM_RAW_DIR = BASE_DIR / "data" / "raw" / "llm" / "nyt_sentiment"
LLM_MODELS = {"claude-haiku-5-5": "Haiku 5.5", "claude-sonnet-5-5": "Sonnet 5.5", "claude-opus-5-5": "Opus 5.5"}
LLM_PRICES = {"claude-haiku-5-5": (0.10, 0.50), "claude-sonnet-5-5": (2.00, 10.00), "claude-opus-5-5": (4.00, 20.00)}  # $ per 1M token (input, output), listino 2026-10
LLM_EFFORT = "low"
LLM_MAX_TOKENS = 4000
LLM_WORKERS = 4
N_DEV_PER_PERIOD = 13

LLM_SYSTEM_PROMPT = """You are annotating New York Times articles about U.S. Immigration and Customs Enforcement (ICE) for a research project. For each article you receive the newspaper section and a short text (headline, abstract and, when available, the first paragraph). Assign two labels, following these guidelines.

TONE: how emotionally charged the text is, regardless of ICE.
- neg: negative events or language (deaths, violent arrests, fear, protests, accusations).
- neu: procedural or informative reporting (data, appointments, court rulings reported without emphasis).
- pos: positive outcomes (release, reunification, success, relief).

STANCE TOWARD ICE: the judgment on the agency expressed by the newspaper itself, not by the people it quotes.
- crit: the newspaper frames ICE negatively in its own voice. Signs: loaded words chosen by the newspaper in the headline or text (for example "dragnet", "roundup", "terrorize", "secretive"); findings of the newspaper's own investigation that ICE did something wrong; a selection of details that presents ICE's conduct as harmful or abusive; an opinion piece or editorial against the agency.
- neu: ICE is an actor or background described in neutral words. Criticism attributed to sources (lawyers, advocates, officials, a report) is neutral when the newspaper does not adopt it. Also neu when ICE is only mentioned and nothing can be inferred.
- fav: the newspaper frames ICE positively or legitimizes its action (security, effectiveness, an opinion piece in favor).

Tone and stance are independent: a text about dramatic events can have a neutral stance, and a calm text can be critical. When sources are quoted, still check whether the newspaper's own wording or choice of details takes a side. Choose neu when the text gives no basis to decide.

Also say whether ICE is central to the text (ice_central = true) or only background or a passing mention (false).

In "evidence", quote the words or facts from the text that determined the stance label, or write "none" if nothing in the text expresses a judgment on ICE."""

LLM_SCHEMA = {
    "type": "object",
    "properties": {
        "evidence": {"type": "string"},
        "tone": {"type": "string", "enum": ["neg", "neu", "pos"]},
        "stance": {"type": "string", "enum": ["crit", "neu", "fav"]},
        "ice_central": {"type": "boolean"},
    },
    "required": ["evidence", "tone", "stance", "ice_central"],
    "additionalProperties": False,
}
LLM_FALLBACK = {"claude-haiku-5-5": False, "claude-sonnet-5-5": True, "claude-opus-5-5": True}

load_dotenv(BASE_DIR / ".env")
llm_client = anthropic.Anthropic(api_key=os.environ["CLAUDE_API_KEY"], max_retries=5)
logging.getLogger("httpx2").setLevel(logging.WARNING)

dev_ids = set(pd.concat([g.sample(N_DEV_PER_PERIOD, random_state=SEED) for _, g in gold.groupby("period")])["id"])
gold["split"] = np.where(gold["id"].isin(dev_ids), "dev", "test")
pd.crosstab(gold["period"], gold["split"])

articoli del gold set per periodo, divisi tra messa a punto (*dev*, 13 per periodo) e test (*test*, 25 per periodo).

In [ ]:
def llm_user_message(row: pd.Series) -> str:
    return f"Section: {row['section_name'] or 'unknown'}\n\nText: {row['semantic_text']}"


def llm_classify(model: str, row: pd.Series, system: str, schema: dict, fallback: bool, message_fn=llm_user_message) -> dict:
    request = dict(
        model=model,
        max_tokens=LLM_MAX_TOKENS,
        system=system,
        output_config={"effort": LLM_EFFORT, "format": {"type": "json_schema", "schema": schema}},
        messages=[{"role": "user", "content": message_fn(row)}],
    )
    if fallback:
        # se il modello rifiuta, l'API ripete la richiesta sul modello che Anthropic indica per quel tipo di rifiuto;
        # il modello che ha risposto davvero resta in response["model"]. Il riassunto del ragionamento interno
        # (display "summarized") finisce nel raw
        response = llm_client.beta.messages.create(betas=["server-side-fallback-2026-07-01"], fallbacks="default",
                                                   thinking={"type": "adaptive", "display": "summarized"}, **request)
    else:
        response = llm_client.messages.create(**request)
    return {"id": row["id"], "model": model, "request_id": response._request_id,
            "called_at": datetime.now(timezone.utc).isoformat(), "response": response.to_dict()}


def llm_raw_path(phase: str, model: str, run: int) -> Path:
    return LLM_RAW_DIR / phase / f"{model}_run{run}.jsonl"


def run_llm(df: pd.DataFrame, phase: str, model: str, run: int,
            system: str = LLM_SYSTEM_PROMPT, schema: dict = LLM_SCHEMA, fallback: bool = False, message_fn=llm_user_message) -> Path:
    """Classifica gli articoli di df non ancora nel file raw; ogni risposta è scritta appena arriva (checkpoint)."""
    path = llm_raw_path(phase, model, run)
    path.parent.mkdir(parents=True, exist_ok=True)
    done = {json.loads(line)["id"] for line in path.open()} if path.exists() else set()
    todo = df[~df["id"].isin(done)]
    log.info("%s %s run %d: %d già fatti, %d da classificare", phase, model, run, len(done), len(todo))
    if todo.empty:
        return path
    with path.open("a") as f, ThreadPoolExecutor(LLM_WORKERS) as pool:
        futures = {pool.submit(llm_classify, model, row, system, schema, fallback, message_fn): row["id"] for _, row in todo.iterrows()}
        for fut in as_completed(futures):
            try:
                f.write(json.dumps(fut.result()) + "\n")
                f.flush()
            except anthropic.BadRequestError as e:
                # es. crediti esauriti: le richieste successive fallirebbero allo stesso modo
                log.error("Richiesta rifiutata (%s): %s. Interrompo.", futures[fut], e.message)
                pool.shutdown(cancel_futures=True)
                break
            except (anthropic.APIStatusError, anthropic.APIConnectionError) as e:
                log.warning("Articolo %s non classificato (%s): riprovato alla prossima esecuzione", futures[fut], e)
    return path


def load_llm_raw(phase: str) -> pd.DataFrame:
    """Una riga per (articolo, modello, run) con etichette, token e costo; stop_reason diverso da end_turn = etichette mancanti."""
    rows = []
    for path in sorted((LLM_RAW_DIR / phase).glob("*.jsonl")):
        model, run = path.stem.rsplit("_run", 1)
        for line in path.open():
            rec = json.loads(line)
            resp = rec["response"]
            labels = {}
            if resp["stop_reason"] == "end_turn":
                labels = json.loads(next(b["text"] for b in resp["content"] if b["type"] == "text"))
            price_in, price_out = LLM_PRICES[model]
            usage = resp["usage"]
            rows.append({"id": rec["id"], "model": model, "served_model": resp["model"], "run": int(run), "stop_reason": resp["stop_reason"],
                         **{k: labels.get(k) for k in ("tone", "emotion", "stance", "ice_central", "reasoning", "evidence")},
                         "input_tokens": usage["input_tokens"], "output_tokens": usage["output_tokens"],
                         "cost_usd": (usage["input_tokens"] * price_in + usage["output_tokens"] * price_out) / 1e6})
    return pd.DataFrame(rows)


dev = gold[gold["split"] == "dev"]
# fase "dev_v2": la cartella "dev" contiene le risposte della prima versione del prompt
for model in LLM_MODELS:
    run_llm(dev, "dev_v2", model, run=1, fallback=LLM_FALLBACK[model])

llm_dev = load_llm_raw("dev_v2")
(llm_dev.groupby("model")
 .agg(articoli=("id", "size"), non_conclusi=("stop_reason", lambda s: (s != "end_turn").sum()),
      risposti_da_altro_modello=("served_model", lambda s: (s != llm_dev.loc[s.index, "model"]).sum()),
      token_output_mediani=("output_tokens", "median"), costo_usd=("cost_usd", "sum"))
 .rename(index=LLM_MODELS).round(3))

*non_conclusi* = risposte senza etichette (rifiuto o risposta troncata) *risposti_da_altro_modello* = articoli su cui il modello ha rifiutato e ha risposto quello di riserva (fallback); *token_output_mediani* = token scritti dal modello (ragionamento interno + JSON) per articolo;

In [ ]:
LLM_TONE = {"neg": "negative", "neu": "neutral", "pos": "positive"}

def build_eval(llm_runs: pd.DataFrame, frame: pd.DataFrame, run: int = 1) -> pd.DataFrame:
    """Articoli del gold set con le etichette di ogni modello (una passata) e i riferimenti della §1.3."""
    dev_eval = frame.reset_index(drop=True).copy()
    dev_eval["sample_weight"] = dev_eval["period"].map(period_sizes) / dev_eval["period"].map(dev_eval["period"].value_counts())

    for model in LLM_MODELS:
        pred = llm_runs[(llm_runs["model"] == model) & (llm_runs["run"] == run)].set_index("id")
        dev_eval[f"llm_tone_{model}"] = dev_eval["id"].map(pred["tone"]).map(LLM_TONE)
        dev_eval[f"llm_stance_{model}"] = dev_eval["id"].map(pred["stance"])
    missing = dev_eval.filter(like="llm_").isna().sum()
    if missing.any():
        log.warning("Etichette mancanti, contate come neutre: %s", missing[missing > 0].to_dict())
        dev_eval[missing.index] = dev_eval[missing.index].fillna({c: "neutral" if "tone" in c else "neu" for c in missing.index})

    # Riferimenti sugli stessi 52 articoli: RoBERTa (§1.3) e la classe più frequente
    dev_eval["roberta_tone"] = dev_eval["id"].map(gold_eval.set_index("id")["roberta_tone"])
    dev_eval["baseline_stance"] = gold["gold_stance"].mode()[0]
    dev_eval["baseline_tone"] = dev_eval["gold_tone"].mode()[0]
    return dev_eval


def llm_scores(dev_eval: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    # con 13 articoli per periodo alcuni ricampionamenti hanno una sola classe: kappa indefinito (NaN), avviso soppresso
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UndefinedMetricWarning)
        return (validation_scores(dev_eval, "gold_stance", LLM_STANCE_MODELS, STANCE_LABELS),
                validation_scores(dev_eval, "gold_tone", LLM_TONE_MODELS, TONE_LABELS))


def overall_table(stance_scores: pd.DataFrame, tone_scores: pd.DataFrame) -> pd.DataFrame:
    return pd.concat({
        "posizione": stance_scores[stance_scores["period"] == "All (weighted)"].pivot(index="model", columns="metric", values="value").loc[list(LLM_STANCE_MODELS.values())],
        "tono": tone_scores[tone_scores["period"] == "All (weighted)"].pivot(index="model", columns="metric", values="value").loc[list(LLM_TONE_MODELS.values())],
    })


LLM_STANCE_MODELS = {f"llm_stance_{m}": n for m, n in LLM_MODELS.items()} | {"baseline_stance": "Majority baseline"}
LLM_TONE_MODELS = {f"llm_tone_{m}": n for m, n in LLM_MODELS.items()} | {"roberta_tone": "RoBERTa", "baseline_tone": "Majority baseline"}
dev_eval = build_eval(llm_dev, dev)
dev_stance_scores, dev_tone_scores = llm_scores(dev_eval)
overall_table(dev_stance_scores, dev_tone_scores).round(2)

metriche sui 52 articoli di messa a punto, pesate sul corpus. Sopra la posizione, sotto il tono. *Majority baseline* risponde sempre con la classe più frequente. Con 52 articoli gli intervalli sono larghi (grafico sotto): la tabella serve a mettere a punto il prompt, non è il risultato finale.

In [ ]:
plot_validation(dev_stance_scores,
                {"Haiku 5.5": PERIOD_COLORS[2], "Sonnet 5.5": PERIOD_COLORS[3], "Opus 5.5": PERIOD_COLORS[0], "Majority baseline": MUTED_COLOR},
                "Stance toward ICE: LLM vs revised manual labels, 52 dev articles (95% bootstrap intervals)", "sentiment_llm_dev_stance.png")

In [ ]:
stance_order = ["crit", "neu", "fav"]
stance_names = {"crit": "critical", "neu": "neutral", "fav": "favorable"}
fig, axes = plt.subplots(1, len(LLM_MODELS), figsize=(11, 3.6))
for ax, (model, name) in zip(axes, LLM_MODELS.items()):
    cm = confusion_matrix(dev_eval["gold_stance"], dev_eval[f"llm_stance_{model}"], labels=stance_order)
    ax.imshow(cm, cmap="Blues")
    for i in range(3):
        for j in range(3):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "#1f1f1f")
    ax.set_xticks(range(3), [stance_names[l] for l in stance_order])
    ax.set_yticks(range(3), [stance_names[l] for l in stance_order])
    ax.set_xlabel("Model")
    ax.set_title(name, fontsize=10)
    ax.grid(False)
axes[0].set_ylabel("Manual label")
fig.suptitle("Stance confusion matrices, 52 dev articles (unweighted)")
fig.tight_layout()
fig.savefig(PLOTS_DIR / "sentiment_llm_dev_stance_confusion.png", dpi=150)
plt.show()

**Come leggere**: righe = posizione manuale, colonne = posizione del modello, numeri = articoli (non pesati). La diagonale sono gli accordi. Le celle da guardare sono *critical → neutral* (critiche del giornale non riconosciute) e *neutral → critical* (cronaca letta come critica).

In [ ]:
stance_cols = [f"llm_stance_{m}" for m in LLM_MODELS]
disagree = dev_eval[dev_eval[stance_cols].ne(dev_eval["gold_stance"], axis=0).any(axis=1)]
evidence = llm_dev[llm_dev["model"] == "claude-opus-5-5"].set_index("id")["evidence"]
pd.DataFrame({
    "periodo": disagree["period"],
    "sezione": disagree["section_name"],
    "testo": disagree["semantic_text"].str.slice(0, 160),
    "gold": disagree["gold_stance"],
    **{LLM_MODELS[m]: disagree[f"llm_stance_{m}"] for m in LLM_MODELS},
    "evidenza Opus": disagree["id"].map(evidence),
}).sort_values("periodo")

Una riga per articolo di messa a punto su cui almeno un modello dà una posizione diversa dal gold set; a destra le parole del testo che Opus indica come decisive ("none" = nessun giudizio su ICE nel testo). Serve a capire se l'errore è del prompt (regola applicata male), un caso al limite o un'informazione che non è nel testo breve.

#### Risultati della prova

- **Posizione**: kappa 0,41 Haiku, 0,38 Sonnet, 0,33 Opus (pesati). Dei 15 articoli critici per l'annotatore ne riconoscono 9-10; in cambio 6-7 dei 36 neutri sono letti come critici.
- **Tono: tutti e tre gli LLM superano nettamente RoBERTa** (kappa 0,49-0,63 contro 0,24 sugli stessi articoli).
- **Nessun rifiuto.**
- **I modelli sono vicini tra loro**: con 52 articoli le differenze non sono significative. Haiku costa circa 30 volte meno di Opus.
- **Disaccordi rimasti**: editoriali, lettere e inchieste del Times che i modelli leggono come critici e il gold set come neutri (le linee guida li classificherebbero critici); cronaca in cui la critica sta nella scelta dei fatti, che i modelli leggono come neutra. Le etichette non vengono cambiate dopo aver visto i modelli: da dichiarare come limite.

**Prossimo passo**: prompt congelato, test sui 100 articoli tenuti da parte (3 modelli × 3 passate), scelta del modello solo sul test.

### 1.5 Test sui 100 articoli tenuti da parte

I 100 articoli di test (25 per periodo) non sono mai stati usati per metterlo a punto, quindi le metriche di questa sezione sono quelle da riportare.

**Tre passate per modello.** La temperatura non è impostabile, quindi lo stesso articolo può ricevere etichette diverse in chiamate diverse. Ogni modello classifica i 100 articoli tre volte (`run` 1-3, file raw separati). Si misurano:
- le **metriche per passata** (accuratezza, F1 macro, kappa pesati, come nella §1.3): quanto cambia il risultato da una passata all'altra;
- la **stabilità**: quota di articoli con la stessa etichetta in tutte e tre le passate e kappa medio tra coppie di passate.

**Criterio di scelta, fissato prima di vedere i risultati.**
1. Misura principale: **kappa pesato sulla posizione, media delle tre passate**; è la misura che serve per il confronto con Telegram.
2. Se l'intervallo bootstrap al 95% del modello migliore (passata 1) contiene il kappa di un modello più economico, le differenze non sono distinguibili: si sceglie il **più economico** fra quelli dentro l'intervallo, perché il modello va poi applicato a 2.093 articoli e a molti commenti.
3. Tono e stabilità servono da controllo: un modello con stabilità molto più bassa degli altri (meno dell'80% degli articoli con la stessa posizione nelle tre passate) non viene scelto.

In [ ]:
N_TEST_RUNS = 3
test = gold[gold["split"] == "test"]
for model in LLM_MODELS:
    for run in range(1, N_TEST_RUNS + 1):
        run_llm(test, "test", model, run=run, fallback=LLM_FALLBACK[model])

llm_test = load_llm_raw("test")
(llm_test.groupby("model")
 .agg(risposte=("id", "size"), non_conclusi=("stop_reason", lambda s: (s != "end_turn").sum()),
      risposti_da_altro_modello=("served_model", lambda s: (s != llm_test.loc[s.index, "model"]).sum()),
      token_output_mediani=("output_tokens", "median"), costo_usd=("cost_usd", "sum"))
 .rename(index=LLM_MODELS).round(3))

In [ ]:
test_evals = {run: build_eval(llm_test, test, run) for run in range(1, N_TEST_RUNS + 1)}
# bootstrap solo sulla passata 1 (per l'intervallo del criterio 2); per le altre passate bastano i valori puntuali
test_stance_scores, test_tone_scores = llm_scores(test_evals[1])


def overall_point(frame: pd.DataFrame, gold_col: str, models: dict[str, str], labels: list[str]) -> dict[str, float]:
    return {name: classification_metrics(frame[gold_col], frame[col], labels, frame["sample_weight"])["kappa"] for col, name in models.items()}


test_kappa = pd.DataFrame([
    {"run": run, "task": task, "model": name, "kappa": k}
    for run, frame in test_evals.items()
    for task, gold_col, models, labels in [("stance", "gold_stance", LLM_STANCE_MODELS, STANCE_LABELS), ("tone", "gold_tone", LLM_TONE_MODELS, TONE_LABELS)]
    for name, k in overall_point(frame, gold_col, models, labels).items()
])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
model_order = list(LLM_MODELS.values()) + ["RoBERTa", "Majority baseline"]
for ax, (task, title) in zip(axes, [("stance", "Stance toward ICE"), ("tone", "Tone")]):
    s = test_kappa[test_kappa["task"] == task]
    for k, name in enumerate(model_order):
        v = s.loc[s["model"] == name, "kappa"]
        if v.empty:
            continue
        color = MUTED_COLOR if name in ("RoBERTa", "Majority baseline") else MAIN_COLOR
        ax.scatter(v, [k] * len(v), color=color, s=30, alpha=0.8, zorder=3)
        ax.plot([v.mean()] * 2, [k - 0.3, k + 0.3], color="#1f1f1f", lw=1.5)
    ci = (test_stance_scores if task == "stance" else test_tone_scores)
    ci = ci[(ci["period"] == "All (weighted)") & (ci["metric"] == "kappa")].set_index("model")
    for k, name in enumerate(model_order):
        if name in ci.index:
            ax.hlines(k, ci.at[name, "lo"], ci.at[name, "hi"], color=MUTED_COLOR, lw=4, alpha=0.5, zorder=1)
    ax.set_title(title)
    ax.set_xlabel("Weighted Cohen's kappa")
    ax.set_xlim(-0.1, 1)
axes[0].set_yticks(range(len(model_order)), model_order)
axes[0].invert_yaxis()
fig.suptitle("Test set (100 held-out articles): kappa per run (dots), mean (bar), 95% bootstrap interval of run 1 (grey band)")
fig.tight_layout()
fig.savefig(PLOTS_DIR / "sentiment_llm_test_kappa.png", dpi=150, bbox_inches="tight")
plt.show()

(test_kappa[test_kappa["model"].isin(LLM_MODELS.values())]
 .pivot_table(index="model", columns="task", values="kappa", aggfunc=["mean", "min", "max"])
 .loc[list(LLM_MODELS.values())].round(2))

Punti vicini tra loro vogliono dire risultati stabili tra le passate. Se le bande di due modelli si sovrappongono molto, la differenza tra i due non è distinguibile. La tabella riporta media, minimo e massimo delle tre passate.

In [ ]:
def run_stability(model: str, task: str) -> dict[str, float]:
    col = f"llm_{task}_{model}"
    labels_by_run = pd.DataFrame({run: frame[col] for run, frame in test_evals.items()})
    pairs = [(a, b) for a in labels_by_run for b in labels_by_run if a < b]
    return {"same_label_all_runs": labels_by_run.nunique(axis=1).eq(1).mean(),
            "mean_pairwise_kappa": np.mean([cohen_kappa_score(labels_by_run[a], labels_by_run[b]) for a, b in pairs])}


stability = pd.DataFrame([{"model": name, "task": task, **run_stability(model, task)}
                          for model, name in LLM_MODELS.items() for task in ("stance", "tone")])

fig, ax = plt.subplots(figsize=(8, 2.8))
y = np.arange(len(LLM_MODELS))
for k, (task, color) in enumerate([("stance", MAIN_COLOR), ("tone", PERIOD_COLORS[1])]):
    s = stability[stability["task"] == task].set_index("model").loc[list(LLM_MODELS.values())]
    ax.barh(y + (k - 0.5) * 0.35, s["same_label_all_runs"], height=0.35, color=color, label=task)
    for yy, v in zip(y + (k - 0.5) * 0.35, s["same_label_all_runs"]):
        ax.text(v + 0.01, yy, f"{v:.0%}", va="center", fontsize=8)
ax.axvline(0.8, color="#1f1f1f", lw=0.8, ls="--")
ax.set_yticks(y, list(LLM_MODELS.values()))
ax.invert_yaxis()
ax.set_xlim(0, 1.08)
ax.set_xlabel("Share of test articles with the same label in all 3 runs")
ax.legend(loc="lower right", fontsize=8, frameon=False)
ax.set_title("Run-to-run stability on the test set")
fig.tight_layout()
fig.savefig(PLOTS_DIR / "sentiment_llm_test_stability.png", dpi=150)
plt.show()
stability.pivot(index="model", columns="task", values="mean_pairwise_kappa").loc[list(LLM_MODELS.values())].round(2)

#### Risultati del test
- **Posizione**: kappa pesato medio sulle tre passate **Haiku 0,41** (0,36-0,47 tra le passate; intervallo al 95% della passata 1: 0,15-0,59), Opus 0,28, Sonnet 0,16. Accuratezza pesata di Haiku 0,69 contro 0,56 del riferimento minimo (sempre "neutra").
- **Scelta (criterio fissato prima del test): Haiku 5.5.** È il migliore sulla misura principale ed è anche il più economico, quindi i criteri 1 e 2 portano allo stesso modello. Stabilità (criterio 3): il 95% degli articoli riceve la stessa posizione in tutte e tre le passate (Sonnet 98%, Opus 93%), sopra la soglia dell'80%.
- **Tono**: kappa medio 0,50 Haiku, 0,53 Opus, 0,47 Sonnet, cioè simili (RoBERTa 0,24 sul dev). Nessun motivo per usare un modello diverso per il tono: Haiku per entrambe le misure.
- **Cosa sbaglia Haiku** (passata 1, 100 articoli non pesati): riconosce 14 dei 30 critici (16 letti come neutri), legge come critici 6 dei 63 neutri e riconosce 1 dei 7 favorevoli. L'errore principale è la **prudenza**: applicato al corpus, Haiku **sottostimerà la posizione critica**. Va dichiarato, e i confronti tra periodi vanno letti come relativi.
- **Per periodo** (25 articoli ciascuno, intervalli molto larghi): kappa 0,46 nel 2003-2016, 0,60 nel 2017-2020, circa 0 nel 2021-2024 e 0,26 nel 2025+. Il 2025+, cioè il 58% del corpus, è il periodo più debole dopo il 2021-2024: i testi del 2025+ sono più corti, perché senza lead.

**Prossimo passo**: Haiku 5.5 con questo prompt sui 2.093 articoli tramite Batch API.

### 1.6 Applicazione al corpus: Haiku 5.5 sui 2.093 articoli (Batch API)

Usiamo modello e prompt scelti in precedenza per classificare tutti i 2.093 articoli ICE.

**Perché la Batch API.** Le richieste sono elaborate in modo asincrono a metà prezzo (costo stimato circa 0,2 $). Di solito il lotto finisce in meno di un'ora, al massimo in 24 ore. Il fallback lato server non è ammesso nei batch, ma Haiku non lo usava già nel test.

**Raw e checkpoint** (`data/raw/llm/nyt_sentiment/corpus/`):
- `batches.json`: id di ogni lotto inviato con l'elenco dei suoi articoli, salvato **subito dopo l'invio**. Se la cella si interrompe, alla riesecuzione il lotto già inviato viene ripreso e non inviato di nuovo (non si paga due volte).
- `results_<batch_id>.jsonl`: ogni risultato completo dell'API (anche errori e scadenze), scritto prima del parsing.
- Gli articoli con un risultato `succeeded` non vengono più richiesti. Quelli in errore o scaduti entrano in un nuovo lotto alla riesecuzione.

L'attesa nella cella dura al massimo `BATCH_MAX_WAIT_MIN` minuti: se il lotto non è finito, basta rieseguire la cella più tardi.

In [ ]:
import re
import time

from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request

CORPUS_MODEL = "claude-haiku-5-5"
CORPUS_DIR = LLM_RAW_DIR / "corpus"
BATCH_STATE = CORPUS_DIR / "batches.json"
BATCH_POLL_SECONDS = 60
BATCH_MAX_WAIT_MIN = 120


def to_custom_id(article_id: str) -> str:
    cid = article_id.split("//", 1)[1].replace("/", "-")
    assert re.fullmatch(r"[A-Za-z0-9_-]{1,64}", cid), article_id
    return cid


def load_batch_state() -> list[dict]:
    return json.loads(BATCH_STATE.read_text()) if BATCH_STATE.exists() else []


def succeeded_ids() -> set[str]:
    done = set()
    for path in CORPUS_DIR.glob("results_*.jsonl"):
        for line in path.open():
            rec = json.loads(line)
            if rec["result"]["type"] == "succeeded":
                done.add(rec["id"])
    return done


def submit_batch(df: pd.DataFrame) -> dict:
    requests = [Request(custom_id=to_custom_id(row["id"]), params=MessageCreateParamsNonStreaming(
                    model=CORPUS_MODEL, max_tokens=LLM_MAX_TOKENS, system=LLM_SYSTEM_PROMPT,
                    output_config={"effort": LLM_EFFORT, "format": {"type": "json_schema", "schema": LLM_SCHEMA}},
                    messages=[{"role": "user", "content": llm_user_message(row)}]))
                for _, row in df.iterrows()]
    batch = llm_client.messages.batches.create(requests=requests)
    entry = {"batch_id": batch.id, "model": CORPUS_MODEL, "created_at": datetime.now(timezone.utc).isoformat(),
             "ids": df["id"].tolist(), "results_saved": False}
    BATCH_STATE.write_text(json.dumps(load_batch_state() + [entry], indent=1))
    log.info("Lotto %s inviato: %d articoli", batch.id, len(requests))
    return entry


def save_results(entry: dict) -> None:
    by_cid = {to_custom_id(i): i for i in entry["ids"]}
    path = CORPUS_DIR / f"results_{entry['batch_id']}.jsonl"
    with path.open("w") as f:
        for r in llm_client.messages.batches.results(entry["batch_id"]):
            f.write(json.dumps({"id": by_cid[r.custom_id], "custom_id": r.custom_id, "batch_id": entry["batch_id"],
                                "result": r.result.to_dict()}) + "\n")
    state = load_batch_state()
    for e in state:
        if e["batch_id"] == entry["batch_id"]:
            e["results_saved"] = True
    BATCH_STATE.write_text(json.dumps(state, indent=1))
    log.info("Risultati del lotto %s salvati in %s", entry["batch_id"], path.relative_to(BASE_DIR))


def run_corpus_batch(df: pd.DataFrame) -> None:
    CORPUS_DIR.mkdir(parents=True, exist_ok=True)
    pending = [e for e in load_batch_state() if not e["results_saved"]]
    if not pending:
        todo = df[~df["id"].isin(succeeded_ids())]
        log.info("Corpus: %d articoli già classificati, %d da inviare", len(df) - len(todo), len(todo))
        if todo.empty:
            return
        pending = [submit_batch(todo)]
    deadline = time.monotonic() + BATCH_MAX_WAIT_MIN * 60
    for entry in pending:
        while True:
            batch = llm_client.messages.batches.retrieve(entry["batch_id"])
            if batch.processing_status == "ended":
                save_results(entry)
                break
            if time.monotonic() > deadline:
                log.warning("Lotto %s non ancora finito (%s): rieseguire la cella più tardi", entry["batch_id"], batch.request_counts)
                return
            time.sleep(BATCH_POLL_SECONDS)


run_corpus_batch(articles)


def load_corpus_results() -> pd.DataFrame:
    rows = []
    for path in sorted(CORPUS_DIR.glob("results_*.jsonl")):
        for line in path.open():
            rec = json.loads(line)
            res = rec["result"]
            row = {"id": rec["id"], "batch_id": rec["batch_id"], "result_type": res["type"]}
            if res["type"] == "succeeded":
                msg = res["message"]
                row["stop_reason"] = msg["stop_reason"]
                if msg["stop_reason"] == "end_turn":
                    row |= json.loads(next(b["text"] for b in msg["content"] if b["type"] == "text"))
                price_in, price_out = LLM_PRICES[CORPUS_MODEL]
                # Batch API: metà del prezzo standard
                row["cost_usd"] = (msg["usage"]["input_tokens"] * price_in + msg["usage"]["output_tokens"] * price_out) / 1e6 / 2
            rows.append(row)
    out = pd.DataFrame(rows)
    # un articolo può avere un errore in un lotto e un successo in uno successivo: tiene il successo
    out["_ok"] = out["result_type"].eq("succeeded")
    return out.sort_values("_ok").drop_duplicates("id", keep="last").drop(columns="_ok")


corpus_llm = load_corpus_results()
labelled = corpus_llm["stance"].notna() if "stance" in corpus_llm else pd.Series(False, index=corpus_llm.index)
log.info("Corpus: %d/%d articoli con etichette; risultati %s; costo %.3f $", labelled.sum(), len(articles),
         corpus_llm["result_type"].value_counts().to_dict(), corpus_llm.get("cost_usd", pd.Series(dtype=float)).sum())

article_sentiment = articles[["id", "pub_date", "year", "period", "section_name"]].merge(
    corpus_llm.rename(columns={"tone": "llm_tone", "stance": "llm_stance", "ice_central": "llm_ice_central", "evidence": "llm_evidence"})
    [["id", "llm_tone", "llm_stance", "llm_ice_central", "llm_evidence", "stop_reason", "batch_id"]],
    on="id", how="left")
article_sentiment["llm_model"] = CORPUS_MODEL
article_sentiment.to_parquet(OUT_ARTICLE_SENTIMENT, index=False)
log.info("Scritto %s (%d righe)", OUT_ARTICLE_SENTIMENT.relative_to(BASE_DIR), len(article_sentiment))

# Controllo: sui 100 articoli di test, il lotto dà le stesse etichette delle tre passate del test?
check = test[["id"]].merge(article_sentiment[["id", "llm_stance"]], on="id")
same = {run: (check["llm_stance"].values == check["id"].map(llm_test[(llm_test["model"] == CORPUS_MODEL) & (llm_test["run"] == run)].set_index("id")["stance"]).values).mean()
        for run in range(1, N_TEST_RUNS + 1)}
log.info("Posizione del lotto uguale a quella delle passate di test: %s", {k: f"{v:.0%}" for k, v in same.items()})

### 1.7 Tono e posizione per periodo

Quote delle etichette di Haiku sull'intero corpus, per periodo. Va però ricordato quanto emerso nel test (§1.5): Haiku **sottostima la posizione critica** (riconosce circa metà dei critici) e nel 2021-2024 e nel 2025+ è meno affidabile. Le quote si leggono come confronto **relativo** tra periodi, non come valori assoluti.

In [ ]:
trend = article_sentiment.dropna(subset=["llm_stance"])
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True)
panels = [("llm_stance", ["crit", "neu", "fav"], {"crit": "critical", "neu": "neutral", "fav": "favorable"},
           {"crit": LABEL_COLORS["negative"], "neu": LABEL_COLORS["neutral"], "fav": LABEL_COLORS["positive"]}, "Stance toward ICE"),
          ("llm_tone", ["neg", "neu", "pos"], {"neg": "negative", "neu": "neutral", "pos": "positive"},
           {"neg": LABEL_COLORS["negative"], "neu": LABEL_COLORS["neutral"], "pos": LABEL_COLORS["positive"]}, "Tone")]
for ax, (col, order, names, colors, title) in zip(axes, panels):
    share = pd.crosstab(trend["period"], trend[col], normalize="index").reindex(columns=order, fill_value=0).loc[labels]
    left = np.zeros(len(share))
    for lab in order:
        ax.barh(range(len(share)), share[lab], left=left, color=colors[lab], label=names[lab])
        for y, (l, v) in enumerate(zip(left, share[lab])):
            if v >= 0.06:
                ax.text(l + v / 2, y, f"{v:.0%}", ha="center", va="center", fontsize=8, color="white" if lab != "neu" else "#1f1f1f")
        left += share[lab].to_numpy()
    ax.set_xlim(0, 1)
    ax.set_title(title)
    ax.set_xlabel("Share of ICE articles in the period")
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.42), ncol=3, fontsize=8, frameon=False)
    ax.grid(False)
n_by_period = trend["period"].value_counts()
axes[0].set_yticks(range(len(labels)), [f"{p} (n={n_by_period[p]})" for p in labels])
axes[0].invert_yaxis()
fig.suptitle("NYT ICE articles: stance and tone by period (Claude Haiku 5.5)")
fig.tight_layout()
fig.savefig(PLOTS_DIR / "sentiment_corpus_by_period.png", dpi=150, bbox_inches="tight")
plt.show()

#### Risultati sul corpus

- **Esecuzione**: 2.093 riuscite, 0 rifiuti o risposte troncate. Sui 100 articoli di test la posizione del lotto coincide con quella di ciascuna delle tre passate del test nel 97% dei casi: il lotto si comporta come il test.
- **Posizione critica quasi costante**: 23% nel 2003-2016, 24% nel 2017-2020, 15% nel 2021-2024, 19% nel 2025+. Favorevoli quasi assenti (meno dell'1%). 
- **Tono**: negativo 55% / 65% / 44% / 56%. Il picco è nel 2017-2020 (separazione delle famiglie), poi il 2025+ torna al livello del 2003-2016.
- **Per sezione**: Opinion 49% critici, Magazine 33%, New York 19%, U.S. 16%, World 13%. La posizione critica sta soprattutto negli editoriali e nei commenti, come previsto dalle linee guida (editoriale contrario = `crit`); la cronaca resta per lo più neutra.
- **Cautele**: Haiku sottostima i critici (14 su 30 nel test), quindi le quote vere sono più alte in tutti i periodi. Le differenze tra periodi sono piccole rispetto all'errore del modello, e nel 2021-2024 (96 articoli, kappa circa 0 nel test) il valore non è affidabile. La lettura sicura è l'**assenza di un aumento netto** della posizione critica nel 2025+, non le singole percentuali.

Output: `data/processed/nyt/articles_ice_sentiment.parquet` (una riga per articolo: `llm_tone`, `llm_stance`, `llm_ice_central`, `llm_evidence`, `batch_id`, `llm_model`), `plots/sentiment_corpus_by_period.png`.

### 1.8 Esempi di articoli classificati

Una tabella di esempi per leggere le etichette di Haiku sul corpus. Per ogni combinazione di posizione e tono c'è un articolo per periodo, estratto a caso con seed fisso (`SEED`).

In [ ]:
EXAMPLE_CELLS = [("crit", "neg"), ("crit", "neu"), ("neu", "neg"), ("neu", "neu"), ("neu", "pos"), ("fav", None)]
TEXT_CHARS = 280

pool = article_sentiment.merge(articles[["id", "headline", "semantic_text"]], on="id")
pool = pool[~pool["id"].isin(gold["id"])].assign(period=lambda d: d["period"].astype(str))

rows = []
for stance, tone in EXAMPLE_CELLS:
    cell = pool[(pool["llm_stance"] == stance) & ((pool["llm_tone"] == tone) if tone else True)]
    for period, g in cell.groupby("period"):
        rows.append(g.sample(1, random_state=SEED).assign(cell_size=len(cell)))
examples = pd.concat(rows)

stance_names = {"crit": "critical", "neu": "neutral", "fav": "favorable"}
tone_names = {"neg": "negative", "neu": "neutral", "pos": "positive"}
example_table = pd.DataFrame({
    "stance": examples["llm_stance"].map(stance_names),
    "tone": examples["llm_tone"].map(tone_names),
    "articles in cell": examples["cell_size"],
    "period": examples["period"],
    "section": examples["section_name"],
    "date": examples["pub_date"].dt.date,
    "headline": examples["headline"],
    "text (semantic_text)": examples["semantic_text"].str.slice(0, TEXT_CHARS).where(examples["semantic_text"].str.len() <= TEXT_CHARS,
                                                                              examples["semantic_text"].str.slice(0, TEXT_CHARS) + "…"),
    "evidence (Haiku)": examples["llm_evidence"],
}).reset_index(drop=True)
example_table.to_csv(PROCESSED_DIR / "sentiment_corpus_examples.csv", index=False)

(example_table.style
 .hide(axis="index")
 .set_properties(**{"text-align": "left", "vertical-align": "top", "white-space": "normal"})
 .set_properties(subset=["text (semantic_text)", "evidence (Haiku)"], **{"min-width": "320px"})
 .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}]))

Cosa si vede in questi 22 esempi:
- **le celle grandi seguono le linee guida.** Critici: il giornale sceglie parole cariche ("Thousands Are Targeted as ICE Prepares to Raid", "The Price? 3 Weeks in ICE Custody") o scrive un articolo d'opinione contro ("The Unnecessary Cruelty…", "A Smart Immigration Move at Rikers"). Neutri a tono negativo: eventi duri raccontati senza giudizio su ICE (tear gas a Portland, sentenza sulla Guardia nazionale). Positivi: esiti positivi come i rilasci, con posizione neutra;
- **evidenza e etichetta a volte non concordano.** In due critici (*Ill and in Pain, Detainee Dies in U.S. Hands* e *ICE Spent $700 Million on 7 Warehouses*) l'evidenza stessa dice che la critica è riportata o che il giornale "does not explicitly judge ICE", ma l'etichetta è `crit`. Sono casi al limite, e l'evidenza aiuta a trovarli;
- **favorevoli**: *A Drug War Inside the Government* (editoriale per dare a ICE le indagini antidroga) è un vero favorevole. *The Democrats' Latest Political Misstep* critica i democratici, e l'evidenza lo dice, ma viene etichettato `fav`: errore probabile;
- **ICE solo di passaggio**: newsletter e articoli di altri temi (*Debating the Use of A.I. in Writing*, *Los Angeles Basks in the Dodgers' Glow*) ricevono posizione neutra e evidenza "none" o equivalente: sono corretti, ma ricordano che parte del corpus nomina ICE solo di sfuggita (`llm_ice_central` = falso nel 51% degli articoli).

## 2. Commenti dei lettori

Stessa idea degli articoli: un LLM (Claude Haiku 5.5, il modello scelto nella §1.5) assegna a ogni commento un'**emozione** e una **posizione**.

**Differenze rispetto agli articoli.**
- **Bersaglio della posizione più ampio**: ICE **e la politica restrittiva sull'immigrazione di cui fa parte** (arresti, raid, detenzione, deportazioni, chiusura del confine, limiti all'asilo, cooperazione della polizia con ICE, repressione delle proteste, sanzioni a città santuario e datori di lavoro). I commentatori nominano raramente l'agenzia (l'11% dei commenti di primo livello, il 6% delle risposte) e, leggendo il pilota, molti commenti su asilo, proteste o polizia prendono chiaramente posizione pur senza nominare ICE (decisione dell'utente, 2026-10-19). La posizione misura quindi più un asse "restrizione contro apertura" che un giudizio sulla sola agenzia: il confronto con la posizione degli articoli (bersaglio = ICE, voce del giornale) è asimmetrico e va dichiarato.
- **Quarta etichetta `not_about`**: il commento non tocca immigrazione, immigrati o enforcement (solo Trump come persona, l'economia, il giornale, insulti ad altri lettori). La polarizzazione si misura solo sui commenti pertinenti.
- **Emozione al posto del tono**: nei commenti quasi tutto è negativo, quindi negativo/neutro/positivo distingue poco; un'emozione dominante (rabbia, disprezzo, paura, tristezza, speranza, nessuna) dice di più sulla polarizzazione ed è la stessa scheda da usare per Telegram. Non è confrontabile con il tono degli articoli.
- **Contesto**: il titolo dell'articolo per tutti i commenti; per le risposte anche il commento a cui rispondono, con l'istruzione di etichettare solo la risposta.

### 2.1 Selezione e pulizia

- **Pulizia come nel notebook 09 §5**: si tolgono URL e il nome del lettore a cui si risponde a inizio testo. Le `@menzioni` diventano `@user`, invece di essere cancellate, così il modello sa che il commento si rivolge a qualcuno. Per riconoscere i nomi composti ("@Jane Doe NY") servono i nomi visualizzati, letti da `comments_ice.parquet` **solo in memoria** e cancellati subito dopo: nessun nome di lettore arriva all'API né agli output.
- **Almeno `MIN_COMMENT_WORDS` = 20 parole** dopo la pulizia, senza contare `@user` (decisione dell'utente): restano fuori le battute di poche parole, una parte delle quali ha una posizione netta.
- **Commento padre** per le risposte: stessa pulizia, tagliato a `MAX_PARENT_WORDS` = 250 parole (oltre è contesto ridondante e costo). Il padre può avere meno di 20 parole: serve solo come contesto.

In [ ]:
IN_COMMENTS_RAW = PROCESSED_DIR / "comments_ice.parquet"  # contiene i nomi visualizzati: letto solo in memoria
MIN_COMMENT_WORDS = 20
MAX_PARENT_WORDS = 250
URL_RE = re.compile(r"https?://\S+|www\.\S+")
MAX_NAME_TOKENS = 5


def load_display_names() -> tuple[set, pd.Series]:
    raw = pd.read_parquet(IN_COMMENTS_RAW, columns=["comment_id", "user_display_name", "parent_user_display_name"])
    names = set(raw["user_display_name"].dropna()) | set(raw["parent_user_display_name"].dropna())
    return names, raw.set_index("comment_id")["parent_user_display_name"]


def replace_mentions(text: str, names: set) -> str:
    # stessa ricerca del nome più lungo del notebook 09 (strip_mentions), ma la menzione diventa "@user"
    out, i = [], 0
    while (j := text.find("@", i)) >= 0:
        out.append(text[i:j])
        rest = text[j + 1:]
        tokens = re.split(r"(\s+)", rest)
        cut = None
        for k in range(min(MAX_NAME_TOKENS, (len(tokens) + 1) // 2), 0, -1):
            candidate = "".join(tokens[:2 * k - 1])
            if candidate.rstrip(":,;.!?") in names:
                cut = len(candidate.rstrip(":,;.!?"))
                break
        if cut is None:
            match = re.match(r"[\w.\-]+", rest)
            cut = match.end() if match else 0
        out.append("@user" if cut else "@")
        i = j + 1 + cut
    out.append(text[i:])
    return "".join(out)


def clean_comment(body: str, parent_name: str | None, names: set) -> str:
    text = replace_mentions(body, names) if "@" in body else body
    text = re.sub(r"\s+", " ", URL_RE.sub(" ", text)).strip()
    if isinstance(parent_name, str) and parent_name and text.startswith(parent_name):
        text = text[len(parent_name):]
    return text.strip(" :,;-–—")


comments = pd.read_parquet(IN_COMMENTS, columns=["comment_id", "article_id", "parent_id", "is_reply", "body", "user_key",
                                                 "parent_user_key", "period", "recommendations", "created_at"])
display_names, parent_names = load_display_names()
comments["text"] = [clean_comment(body or "", parent_names.get(cid), display_names)
                    for cid, body in zip(comments["comment_id"], comments["body"])]
del display_names, parent_names
comments = comments.drop(columns="body")
comments["n_words"] = comments["text"].str.replace("@user", " ", regex=False).str.split().str.len().fillna(0).astype(int)

text_by_id = comments.set_index("comment_id")["text"]
comments["parent_text"] = comments["parent_id"].map(text_by_id).fillna("")
comments["parent_text"] = comments["parent_text"].map(lambda t: " ".join(t.split()[:MAX_PARENT_WORDS]) + (" …" if len(t.split()) > MAX_PARENT_WORDS else ""))
comments["headline"] = comments["article_id"].map(articles.set_index("id")["headline"]).fillna("")
comments["id"] = comments["comment_id"].astype(str)
comments["type"] = np.where(comments["is_reply"], "reply", "top-level")

selected = comments[comments["n_words"] >= MIN_COMMENT_WORDS].reset_index(drop=True)
residual = selected["text"].str.contains(r"@(?!user\b)\w", regex=True).sum()
log.info("Commenti: %d, con almeno %d parole: %d (%.0f%%); risposte senza padre trovato: %d; menzioni non sostituite: %d",
         len(comments), MIN_COMMENT_WORDS, len(selected), len(selected) / len(comments) * 100,
         (selected["is_reply"] & selected["parent_text"].eq("")).sum(), residual)
pd.crosstab(selected["period"], selected["type"], margins=True, margins_name="total").loc[labels + ["total"]]

### 2.2 Prompt per i commenti

**Emozione**: una sola, la dominante, tra `anger` (indignazione), `contempt` (disprezzo, scherno, sarcasmo verso qualcuno), `fear` (preoccupazione, allarme), `sadness` (dolore, sconforto, pietà), `hope` (le emozioni positive: speranza, sollievo, soddisfazione, lode, solidarietà), `none` (commento calmo o argomentativo). Le emozioni positive sono rare nei commenti, quindi stanno in una sola classe.

**Posizione** con il bersaglio esteso (§2). Regole per i casi tipici dei commenti:
- **sarcasmo e ironia**: lode finta, iperbole e domande retoriche si leggono per il significato inteso (con esempi nel prompt);
- **`neu` solo se il commento non pende da nessuna parte**: se pende, anche in modo implicito, prende quella parte;
- **`not_about` solo se il commento non tocca affatto l'immigrazione**: asilo, proteste, polizia, confine e politica migratoria sono pertinenti;
- **emozione e posizione indipendenti**: un commento arrabbiato può essere favorevole a ICE;
- una **risposta breve che concorda o dissente** prende la posizione di ciò con cui concorda o dissente.

Formato: JSON a schema fisso con `reasoning` (al massimo 40 parole: sarcasmo, emozione, posizione), `evidence` (le parole del commento che decidono la posizione, oppure "none"), `emotion`, `stance`. `reasoning` viene **prima** delle etichette, così il modello ragiona prima di decidere; è limitato a 40 parole perché senza limite triplicava i token di output (213 in mediana) e il lotto non sarebbe entrato nel budget.

Il prompt è stato messo a punto sul pilota in tre versioni (2026-10-16/19): le prime due davano `not_about` o `neu` a molti commenti su asilo, proteste e polizia o che pendevano da una parte, e leggevano male il sarcasmo; le loro risposte restano in `comments_pilot/` e `comments_pilot_v2/` ma non sono usate.

In [ ]:
COMMENT_SYSTEM_PROMPT = """You are an expert political science annotator labeling reader comments posted under New York Times articles about U.S. Immigration and Customs Enforcement (ICE).

For each comment, you receive the article's headline and, if applicable, the previous comment it replies to. Use the headline and previous comment ONLY as context to understand the target comment. Label ONLY the text marked "COMMENT". Mentions of other readers are replaced with "@user".

You must annotate two independent dimensions: EMOTION and STANCE.

1. EMOTION (the dominant emotion the commenter expresses; choose exactly one):
- anger: outrage, indignation, fury.
- contempt: scorn, mockery, sarcasm, or disgust toward someone.
- fear: worry, anxiety, or alarm about what is happening or may happen.
- sadness: grief, despair, pity, or disappointment.
- hope: positive emotions such as hope, relief, satisfaction, praise, or solidarity.
- none: calm, informative, or argumentative without a clear emotion.
If several emotions are present, choose the strongest one.

2. STANCE TOWARD ICE AND IMMIGRATION ENFORCEMENT
The target is ICE and the restrictive immigration agenda it is part of: arrests, raids, detention, deportations, border closure, limits on asylum and refugees, police cooperation with ICE, crackdowns on protests against ICE, penalties for sanctuary cities or for employers of undocumented workers.
- crit: opposes ICE or restrictive immigration enforcement and policies, or defends immigrants, asylum seekers, sanctuary policies, or protesters against them (e.g., calls to abolish ICE, describing deportations as cruel, defending the right to asylum, supporting protests against raids).
- fav: supports ICE or restrictive immigration enforcement and policies, or rejects the claims of immigrants and their defenders (e.g., "laws must be enforced", "deport them", "close the border", "they broke the law, no sympathy", attacks on sanctuary cities or on protesters).
- neu: discusses the topic but takes no side at all (e.g., a factual remark, a genuine question, a view that explicitly weighs both sides). If the comment leans toward one side, even implicitly, assign that side instead of neu.
- not_about: does not address immigration, immigrants, or enforcement at all (e.g., comments only about Trump's personality, the economy, the newspaper, or insulting other readers). Comments about asylum, protests, police, the border, or immigration policy ARE about the topic.

CRITICAL INSTRUCTIONS:
- SARCASM & IRONY: Pay special attention to fake praise, hyperbole, and rhetorical questions (e.g., "Wow, what brave heroes arresting a grandmother", "Yes, traumatizing toddlers will totally fix the economy"). Look past the literal words and classify the STANCE based on the intended meaning (which in these cases is "crit"). Sarcasm usually signals contempt as the emotion.
- INDEPENDENT DIMENSIONS: Emotion and stance are not the same. A comment can be angry and fav ("I am furious that Democrats want to abolish ICE!"), or hopeful and crit ("Glad the judge stopped these deportations"). Judge them separately.
- REPLIES: A short reply ("Exactly!" or "You are wrong") takes its stance from what it agrees or disagrees with in the previous comment.

OUTPUT FORMAT:
Return ONLY a valid JSON object with the following keys, in this exact order:
{
  "reasoning": "At most 40 words: whether the comment uses sarcasm, which emotion dominates, and which side it takes.",
  "evidence": "Quote the exact words from the COMMENT that justify the stance (or write 'none').",
  "emotion": "anger, contempt, fear, sadness, hope, or none",
  "stance": "crit, fav, neu, or not_about"
}"""

COMMENT_EMOTION_LABELS = ["anger", "contempt", "fear", "sadness", "hope", "none"]
COMMENT_STANCE_LABELS = ["crit", "neu", "fav", "not_about"]
COMMENT_SCHEMA = {
    "type": "object",
    "properties": {
        "reasoning": {"type": "string"},
        "evidence": {"type": "string"},
        "emotion": {"type": "string", "enum": COMMENT_EMOTION_LABELS},
        "stance": {"type": "string", "enum": COMMENT_STANCE_LABELS},
    },
    "required": ["reasoning", "evidence", "emotion", "stance"],
    "additionalProperties": False,
}


def comment_user_message(row: pd.Series) -> str:
    parts = [f"Article headline: {row['headline'] or 'unknown'}"]
    if row["parent_text"]:
        parts.append(f"Previous comment (context only): {row['parent_text']}")
    parts.append(f"COMMENT: {row['text']}")
    return "\n\n".join(parts)


# Esempio del messaggio inviato al modello (una risposta del pool)
print(comment_user_message(selected[selected["is_reply"]].sample(1, random_state=SEED).iloc[0]))

### 2.3 Pilota: stabilità ed esempi

Usiamo COMMENT_PILOT_PER_STRATUM = 38 commenti per ognuno degli 8 strati periodo × tipo (304 in tutto, seed fisso), classificati **due volte** con Haiku (stesse impostazioni degli articoli: effort `low`, structured outputs, niente fallback) al posto di un gold set per guardare:

- **stabilità**: quota di commenti con la stessa posizione nelle due passate e kappa tra le passate. Un'etichetta che cambia da una chiamata all'altra non può misurare la polarizzazione;
- **distribuzione per strato**: quanti `not_about`, quanti favorevoli. Il campione è stratificato, quindi le quote **non** sono quelle del corpus;
- **esempi per classe**, da leggere: se una classe è chiaramente sbagliata si corregge il prompt **prima** del lotto sul corpus, poi lo si congela.

Il pilota non entra nei risultati: il lotto della §2.4 riclassifica tutti i commenti, compresi questi.

In [ ]:
COMMENT_PILOT_PER_STRATUM = 38
COMMENT_MODEL = "claude-haiku-5-5"
COMMENT_PILOT_PHASE = "comments_pilot_v3"  # risposte delle versioni precedenti del prompt: §2.2

comment_pilot = pd.concat([g.sample(min(COMMENT_PILOT_PER_STRATUM, len(g)), random_state=SEED)
                           for _, g in selected.groupby(["period", "type"], observed=True)]).reset_index(drop=True)
for run in (1, 2):
    run_llm(comment_pilot, COMMENT_PILOT_PHASE, COMMENT_MODEL, run=run, system=COMMENT_SYSTEM_PROMPT,
            schema=COMMENT_SCHEMA, message_fn=comment_user_message)

llm_comment_pilot = load_llm_raw(COMMENT_PILOT_PHASE)
pilot_labels = {run: llm_comment_pilot[llm_comment_pilot["run"] == run].set_index("id") for run in (1, 2)}
pilot = comment_pilot.assign(**{f"stance_run{r}": comment_pilot["id"].map(pilot_labels[r]["stance"]) for r in (1, 2)},
                             **{f"emotion_run{r}": comment_pilot["id"].map(pilot_labels[r]["emotion"]) for r in (1, 2)},
                             evidence=comment_pilot["id"].map(pilot_labels[1]["evidence"]),
                             reasoning=comment_pilot["id"].map(pilot_labels[1]["reasoning"]))

stability_rows = []
for group, g in [("all", pilot)] + list(pilot.groupby("type")):
    stability_rows.append({
        "group": group, "comments": len(g),
        "same stance": (g["stance_run1"] == g["stance_run2"]).mean(),
        "stance kappa (run 1 vs 2)": cohen_kappa_score(g["stance_run1"], g["stance_run2"], labels=COMMENT_STANCE_LABELS),
        "same emotion": (g["emotion_run1"] == g["emotion_run2"]).mean(),
        "emotion kappa (run 1 vs 2)": cohen_kappa_score(g["emotion_run1"], g["emotion_run2"], labels=COMMENT_EMOTION_LABELS),
    })
log.info("Pilota: %d risposte, non concluse %d, costo %.3f $, token di output mediani %d", len(llm_comment_pilot),
         (llm_comment_pilot["stop_reason"] != "end_turn").sum(), llm_comment_pilot["cost_usd"].sum(),
         llm_comment_pilot["output_tokens"].median())
pd.DataFrame(stability_rows).set_index("group").round(2)

*same* = quota di commenti con la stessa etichetta nelle due passate; *kappa* = accordo oltre il caso tra le due passate (1 = sempre identiche). Sugli articoli Haiku dava la stessa posizione nelle tre passate nel 95% dei casi. L'emozione ha sei classi invece di tre, quindi ci si aspetta un accordo più basso del tono.

In [ ]:
stance_colors = {"crit": LABEL_COLORS["negative"], "neu": LABEL_COLORS["neutral"], "fav": LABEL_COLORS["positive"], "not_about": "#e6e6e1"}
stance_full = {"crit": "critical", "neu": "neutral", "fav": "favorable", "not_about": "not about immigration"}
strata = [(p, t) for p in labels for t in ("top-level", "reply")]
share = (pd.crosstab([pilot["period"], pilot["type"]], pilot["stance_run1"], normalize="index")
         .reindex(columns=COMMENT_STANCE_LABELS, fill_value=0).reindex(strata))

fig, ax = plt.subplots(figsize=(9, 4))
left = np.zeros(len(share))
for lab in COMMENT_STANCE_LABELS:
    ax.barh(range(len(share)), share[lab], left=left, color=stance_colors[lab], label=stance_full[lab],
            edgecolor="white", linewidth=0.5)
    for y, (l, v) in enumerate(zip(left, share[lab])):
        if v >= 0.07:
            ax.text(l + v / 2, y, f"{v:.0%}", ha="center", va="center", fontsize=8,
                    color="white" if lab in ("crit", "fav") else "#1f1f1f")
    left += share[lab].to_numpy()
ax.set_yticks(range(len(share)), [f"{p} · {t}" for p, t in strata])
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.grid(False)
ax.set_xlabel("Share of pilot comments in the stratum (run 1)")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.32), ncol=4, fontsize=8, frameon=False)
ax.set_title(f"Comment stance in the pilot ({COMMENT_PILOT_PER_STRATUM} comments per stratum; not corpus shares)", loc="left", fontsize=10)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "sentiment_comments_pilot_stance.png", dpi=150, bbox_inches="tight")
plt.show()

Una barra per strato (periodo · tipo), divisa secondo la posizione data da Haiku nella prima passata; grigio chiaro = commenti che non toccano l'immigrazione. Ogni strato ha lo stesso numero di commenti, quindi le barre confrontano gli strati tra loro ma non danno le quote del corpus. Le cose da guardare: quanta parte dei commenti è not_about (sarà esclusa dalle misure di polarizzazione) e quanti favorevoli ci sono (se sono pochissimi la polarizzazione sarà soprattutto critici contro neutri).

In [ ]:
EXAMPLES_PER_CELL = 2
TEXT_CHARS_COMMENT = 400
cut = lambda s, n: s if len(s) <= n else s[:n] + "…"

rows = []
for stance in COMMENT_STANCE_LABELS:
    for typ in ("top-level", "reply"):
        g = pilot[(pilot["stance_run1"] == stance) & (pilot["type"] == typ)]
        rows.append(g.sample(min(EXAMPLES_PER_CELL, len(g)), random_state=SEED))
pilot_examples = pd.concat(rows)
pilot_example_table = pd.DataFrame({
    "stance (run 1 / run 2)": pilot_examples["stance_run1"].map(stance_full) + " / " + pilot_examples["stance_run2"].map(stance_full),
    "emotion": pilot_examples["emotion_run1"],
    "type": pilot_examples["type"],
    "period": pilot_examples["period"].astype(str),
    "article headline": pilot_examples["headline"],
    "previous comment": pilot_examples["parent_text"].map(lambda s: cut(s, 200)),
    "comment": pilot_examples["text"].map(lambda s: cut(s, TEXT_CHARS_COMMENT)),
    "evidence (Haiku)": pilot_examples["evidence"],
    "reasoning (Haiku)": pilot_examples["reasoning"],
}).reset_index(drop=True)

(pilot_example_table.style
 .hide(axis="index")
 .set_properties(**{"text-align": "left", "vertical-align": "top", "white-space": "normal"})
 .set_properties(subset=["comment", "previous comment", "evidence (Haiku)", "reasoning (Haiku)"], **{"min-width": "260px"})
 .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}]))

#### Emozioni nel pilota

Quota di ogni emozione per posizione (prima passata). Serve a controllare che le sei classi siano usate davvero (se una è quasi vuota conviene accorparla) e che emozione e posizione non coincidano (per esempio che non tutti i critici siano `anger`).

In [ ]:
emotion_colors = {"anger": "#c0392b", "contempt": "#8e44ad", "fear": "#e67e22", "sadness": "#2c6fbb", "hope": "#27ae60", "none": "#d5d5d0"}
emo_share = (pd.crosstab(pilot["stance_run1"], pilot["emotion_run1"], normalize="index")
             .reindex(index=COMMENT_STANCE_LABELS, columns=COMMENT_EMOTION_LABELS, fill_value=0))
emo_n = pilot["stance_run1"].value_counts().reindex(COMMENT_STANCE_LABELS, fill_value=0)

fig, ax = plt.subplots(figsize=(9, 3))
left = np.zeros(len(emo_share))
for emo in COMMENT_EMOTION_LABELS:
    ax.barh(range(len(emo_share)), emo_share[emo], left=left, color=emotion_colors[emo], label=emo, edgecolor="white", linewidth=0.5)
    for y, (l, v) in enumerate(zip(left, emo_share[emo])):
        if v >= 0.07:
            ax.text(l + v / 2, y, f"{v:.0%}", ha="center", va="center", fontsize=8, color="#1f1f1f" if emo == "none" else "white")
    left += emo_share[emo].to_numpy()
ax.set_yticks(range(len(emo_share)), [f"{stance_full[s]} (n={emo_n[s]})" for s in COMMENT_STANCE_LABELS])
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.grid(False)
ax.set_xlabel("Share of pilot comments with the stance (run 1)")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.45), ncol=6, fontsize=8, frameon=False)
ax.set_title("Dominant emotion by stance in the pilot (not corpus shares)", loc="left", fontsize=10)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "sentiment_comments_pilot_emotion.png", dpi=150, bbox_inches="tight")
plt.show()

### 2.4 Applicazione al corpus: Haiku 5.5 sui 229.439 commenti (Batch API)

**Prompt caching.** Il prompt di sistema (circa 1.760 token con lo schema, uguale per tutte le richieste) è marcato con `cache_control`: dopo la prima scrittura le richieste lo leggono dalla cache a 0,1× il prezzo di input. TTL di **1 ora** invece dei 5 minuti di default, perché le richieste di un lotto possono essere elaborate a distanza di più di 5 minuti; la scrittura costa 2× invece di 1,25×, ma avviene poche volte. Nei batch la cache è "best effort": il costo vero si legge alla fine dai campi `cache_read_input_tokens` e `cache_creation_input_tokens` di ogni risposta, e il costo calcolato qui li conta (scrittura 1h 2×, scrittura 5 min 1,25×, lettura 0,1×, tutto a metà prezzo per la Batch API).

**Lotti.** Un lotto della Batch API accetta al massimo 100.000 richieste o 256 MB. Ogni richiesta contiene il prompt di sistema (circa 7 KB), quindi i commenti sono divisi in lotti di al massimo `MAX_BATCH_MB` = 200 MB (margine di sicurezza), 6 lotti da circa 38.000 commenti, elaborati in parallelo dall'API.

**Raw e checkpoint** (`data/raw/llm/nyt_sentiment/comments_corpus/`), come per gli articoli (§1.6):
- `batches.json`: id di ogni lotto con l'elenco dei suoi commenti, salvato **subito dopo ogni invio**. Alla riesecuzione i lotti già inviati sono ripresi e non inviati di nuovo;
- `results_<batch_id>.jsonl`: ogni risultato completo dell'API (anche errori e scadenze), scritto prima del parsing;
- i commenti con un risultato `succeeded` non vengono più richiesti; quelli in errore o scaduti entrano in un nuovo lotto alla riesecuzione (con `RUN_COMMENT_BATCH = True`).

**Output**: `comments_ice_sentiment.parquet`, una riga per commento selezionato con le colonne strutturali (articolo, padre, utente pseudonimizzato, periodo, tipo, data, raccomandazioni) e `llm_emotion`, `llm_stance`, `llm_evidence`, `llm_reasoning`. Il testo dei commenti non è copiato: si recupera da `comments_ice_pseudo.parquet` con `comment_id`.

In [ ]:
RUN_COMMENT_BATCH = False  # True solo quando si decide di spendere (~14 $)
COMMENT_BATCH_DIR = LLM_RAW_DIR / "comments_corpus"
COMMENT_BATCH_STATE = COMMENT_BATCH_DIR / "batches.json"
MAX_BATCH_MB = 200
MAX_BATCH_REQUESTS = 100_000
CACHE_TTL = "1h"
# moltiplicatori del prezzo di input per i token in cache
CACHE_READ_MULT, CACHE_WRITE_MULT = 0.1, {"ephemeral_5m_input_tokens": 1.25, "ephemeral_1h_input_tokens": 2.0}

COMMENT_BATCH_PARAMS = dict(
    model=COMMENT_MODEL, max_tokens=LLM_MAX_TOKENS,
    system=[{"type": "text", "text": COMMENT_SYSTEM_PROMPT, "cache_control": {"type": "ephemeral", "ttl": CACHE_TTL}}],
    output_config={"effort": LLM_EFFORT, "format": {"type": "json_schema", "schema": COMMENT_SCHEMA}},
)


def comment_request(row: pd.Series) -> dict:
    return {"custom_id": f"c{row['id']}",
            "params": COMMENT_BATCH_PARAMS | {"messages": [{"role": "user", "content": comment_user_message(row)}]}}


def split_lots(df: pd.DataFrame) -> list[pd.DataFrame]:
    """Lotti consecutivi entro MAX_BATCH_MB e MAX_BATCH_REQUESTS (dimensione = JSON della richiesta)."""
    base = len(json.dumps(comment_request(df.iloc[0]))) - len(json.dumps(comment_user_message(df.iloc[0])))
    sizes = np.array([base + len(json.dumps(comment_user_message(row))) for _, row in df.iterrows()])
    lots, start, total = [], 0, 0
    for i, size in enumerate(sizes):
        if i > start and (total + size > MAX_BATCH_MB * 1e6 or i - start >= MAX_BATCH_REQUESTS):
            lots.append(df.iloc[start:i])
            start, total = i, 0
        total += size
    lots.append(df.iloc[start:])
    return lots


def usage_cost(usage: dict, model: str, batch: bool) -> float:
    """Costo in $ di una risposta, contando i token letti e scritti in cache."""
    price_in, price_out = LLM_PRICES[model]
    writes = usage.get("cache_creation") or {}
    write_tokens = sum(writes.get(k, 0) * m for k, m in CACHE_WRITE_MULT.items())
    if not writes:  # risposta senza dettaglio per TTL: si assume la scrittura più cara
        write_tokens = usage.get("cache_creation_input_tokens", 0) * max(CACHE_WRITE_MULT.values())
    cost = (usage["input_tokens"] * price_in + (usage.get("cache_read_input_tokens") or 0) * CACHE_READ_MULT * price_in
            + write_tokens * price_in + usage["output_tokens"] * price_out) / 1e6
    return cost / 2 if batch else cost


def load_comment_batch_state() -> list[dict]:
    return json.loads(COMMENT_BATCH_STATE.read_text()) if COMMENT_BATCH_STATE.exists() else []


def comment_succeeded_ids() -> set[str]:
    done = set()
    for path in COMMENT_BATCH_DIR.glob("results_*.jsonl"):
        for line in path.open():
            rec = json.loads(line)
            if rec["result"]["type"] == "succeeded":
                done.add(rec["id"])
    return done


def submit_comment_batch(df: pd.DataFrame) -> dict:
    batch = llm_client.messages.batches.create(requests=[comment_request(row) for _, row in df.iterrows()])
    entry = {"batch_id": batch.id, "model": COMMENT_MODEL, "created_at": datetime.now(timezone.utc).isoformat(),
             "ids": df["id"].tolist(), "results_saved": False}
    COMMENT_BATCH_STATE.write_text(json.dumps(load_comment_batch_state() + [entry]))
    log.info("Lotto %s inviato: %d commenti", batch.id, len(df))
    return entry


def save_comment_results(entry: dict) -> None:
    path = COMMENT_BATCH_DIR / f"results_{entry['batch_id']}.jsonl"
    with path.open("w") as f:
        for r in llm_client.messages.batches.results(entry["batch_id"]):
            f.write(json.dumps({"id": r.custom_id[1:], "custom_id": r.custom_id, "batch_id": entry["batch_id"],
                                "result": r.result.to_dict()}) + "\n")
    state = load_comment_batch_state()
    for e in state:
        if e["batch_id"] == entry["batch_id"]:
            e["results_saved"] = True
    COMMENT_BATCH_STATE.write_text(json.dumps(state))
    log.info("Risultati del lotto %s salvati in %s", entry["batch_id"], path.relative_to(BASE_DIR))


def run_comment_batch(df: pd.DataFrame) -> None:
    COMMENT_BATCH_DIR.mkdir(parents=True, exist_ok=True)
    pending = [e for e in load_comment_batch_state() if not e["results_saved"]]
    in_flight = {i for e in pending for i in e["ids"]}
    todo = df[~df["id"].isin(comment_succeeded_ids() | in_flight)]
    lots = split_lots(todo) if len(todo) else []
    log.info("Commenti: %d da inviare in %d lotti, %d in lotti già inviati", len(todo), len(lots), len(in_flight))
    if lots and not RUN_COMMENT_BATCH:
        log.info("RUN_COMMENT_BATCH = False: nessun invio")
    elif lots:
        pending += [submit_comment_batch(lot) for lot in lots]
    deadline = time.monotonic() + BATCH_MAX_WAIT_MIN * 60
    for entry in pending:
        while True:
            batch = llm_client.messages.batches.retrieve(entry["batch_id"])
            if batch.processing_status == "ended":
                save_comment_results(entry)
                break
            if time.monotonic() > deadline:
                log.warning("Lotto %s non ancora finito (%s): rieseguire la cella più tardi", entry["batch_id"], batch.request_counts)
                return
            time.sleep(BATCH_POLL_SECONDS)


# Stima del costo dal pilota v3 (richieste singole, senza cache): la parte fissa (prompt + schema) è
# approssimata con l'input più corto del pilota
fixed_tokens = llm_comment_pilot["input_tokens"].min()
price_in, price_out = LLM_PRICES[COMMENT_MODEL]
n = len(selected)
mean_in, mean_out = llm_comment_pilot["input_tokens"].mean(), llm_comment_pilot["output_tokens"].mean()
est_no_cache = n * (mean_in * price_in + mean_out * price_out) / 1e6 / 2
est_cache = n * ((mean_in - fixed_tokens) * price_in + fixed_tokens * CACHE_READ_MULT * price_in + mean_out * price_out) / 1e6 / 2
log.info("Stima per %d commenti: %.1f $ con la cache, %.1f $ senza (parte fissa ~%d token, output medio %.0f)",
         n, est_cache, est_no_cache, fixed_tokens, mean_out)

run_comment_batch(selected)


def load_comment_results() -> pd.DataFrame:
    rows = []
    for path in sorted(COMMENT_BATCH_DIR.glob("results_*.jsonl")):
        for line in path.open():
            rec = json.loads(line)
            res = rec["result"]
            row = {"id": rec["id"], "batch_id": rec["batch_id"], "result_type": res["type"]}
            if res["type"] == "succeeded":
                msg = res["message"]
                row["stop_reason"] = msg["stop_reason"]
                if msg["stop_reason"] == "end_turn":
                    row |= json.loads(next(b["text"] for b in msg["content"] if b["type"] == "text"))
                row["cache_read_tokens"] = msg["usage"].get("cache_read_input_tokens") or 0
                row["cost_usd"] = usage_cost(msg["usage"], COMMENT_MODEL, batch=True)
            rows.append(row)
    if not rows:
        return pd.DataFrame(columns=["id", "batch_id", "result_type", "stop_reason", "reasoning", "evidence", "emotion", "stance", "cost_usd"])
    out = pd.DataFrame(rows)
    # un commento può avere un errore in un lotto e un successo in uno successivo: tiene il successo
    out["_ok"] = out["result_type"].eq("succeeded")
    return out.sort_values("_ok").drop_duplicates("id", keep="last").drop(columns="_ok")


comment_llm = load_comment_results()
for col in ("stop_reason", "reasoning", "evidence", "emotion", "stance", "cost_usd", "cache_read_tokens"):
    if col not in comment_llm:
        comment_llm[col] = np.nan
log.info("Corpus commenti: %d/%d con etichette; risultati %s; costo %.2f $; richieste con lettura dalla cache %d",
         comment_llm["stance"].notna().sum(), len(selected), comment_llm["result_type"].value_counts().to_dict(),
         comment_llm["cost_usd"].sum(), (comment_llm["cache_read_tokens"] > 0).sum())

if comment_llm["stance"].notna().any():
    comment_sentiment = selected[["comment_id", "id", "article_id", "parent_id", "is_reply", "type", "user_key", "parent_user_key",
                                  "period", "created_at", "recommendations", "n_words"]].merge(
        comment_llm.rename(columns={"emotion": "llm_emotion", "stance": "llm_stance", "evidence": "llm_evidence", "reasoning": "llm_reasoning"})
        [["id", "llm_emotion", "llm_stance", "llm_evidence", "llm_reasoning", "stop_reason", "batch_id"]],
        on="id", how="left").drop(columns="id")
    comment_sentiment["llm_model"] = COMMENT_MODEL
    comment_sentiment.to_parquet(OUT_COMMENT_SENTIMENT, index=False)
    log.info("Scritto %s (%d righe, %d con etichette)", OUT_COMMENT_SENTIMENT.relative_to(BASE_DIR),
             len(comment_sentiment), comment_sentiment["llm_stance"].notna().sum())

### 2.5 Risultati sul corpus

Due grafici su `comments_ice_sentiment.parquet`: la posizione per periodo e tipo di commento, e l'emozione dominante per posizione. Le quote sono sui commenti selezionati (≥ 20 parole), non pesate per articolo: gli articoli con molti commenti, soprattutto nel 2025+, contano di più.

In [ ]:
strata_rows = []
for p_ in labels:
    for t_ in ("top-level", "reply"):
        g = comment_sentiment[(comment_sentiment["period"] == p_) & (comment_sentiment["type"] == t_)]
        strata_rows.append((f"{p_} · {t_} (n={len(g):,})", g["llm_stance"].value_counts(normalize=True)))
corpus_share = pd.DataFrame({k: v for k, v in strata_rows}).T.reindex(columns=COMMENT_STANCE_LABELS, fill_value=0)

fig, ax = plt.subplots(figsize=(9, 4))
left = np.zeros(len(corpus_share))
for lab in COMMENT_STANCE_LABELS:
    ax.barh(range(len(corpus_share)), corpus_share[lab], left=left, color=stance_colors[lab], label=stance_full[lab],
            edgecolor="white", linewidth=0.5)
    for y, (l, v) in enumerate(zip(left, corpus_share[lab])):
        if v >= 0.06:
            ax.text(l + v / 2, y, f"{v:.0%}", ha="center", va="center", fontsize=8,
                    color="white" if lab in ("crit", "fav") else "#1f1f1f")
    left += corpus_share[lab].to_numpy()
ax.set_yticks(range(len(corpus_share)), corpus_share.index)
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.grid(False)
ax.set_xlabel("Share of comments in the period")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.32), ncol=4, fontsize=8, frameon=False)
ax.set_title("Reader comment stance toward ICE and immigration enforcement, by period", loc="left", fontsize=10)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "sentiment_comments_stance_by_period.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
corpus_emo = (pd.crosstab(comment_sentiment["llm_stance"], comment_sentiment["llm_emotion"], normalize="index")
              .reindex(index=COMMENT_STANCE_LABELS, columns=COMMENT_EMOTION_LABELS, fill_value=0))
corpus_emo_n = comment_sentiment["llm_stance"].value_counts().reindex(COMMENT_STANCE_LABELS, fill_value=0)

fig, ax = plt.subplots(figsize=(9, 3))
left = np.zeros(len(corpus_emo))
for emo in COMMENT_EMOTION_LABELS:
    ax.barh(range(len(corpus_emo)), corpus_emo[emo], left=left, color=emotion_colors[emo], label=emo, edgecolor="white", linewidth=0.5)
    for y, (l, v) in enumerate(zip(left, corpus_emo[emo])):
        if v >= 0.06:
            ax.text(l + v / 2, y, f"{v:.0%}", ha="center", va="center", fontsize=8, color="#1f1f1f" if emo == "none" else "white")
    left += corpus_emo[emo].to_numpy()
ax.set_yticks(range(len(corpus_emo)), [f"{stance_full[s]} (n={corpus_emo_n[s]:,})" for s in COMMENT_STANCE_LABELS])
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.grid(False)
ax.set_xlabel("Share of comments with the stance")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.45), ncol=6, fontsize=8, frameon=False)
ax.set_title("Dominant emotion by stance, all selected comments", loc="left", fontsize=10)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "sentiment_comments_emotion_by_stance.png", dpi=150, bbox_inches="tight")
plt.show()

#### Esempi dal corpus

Per controllare le etichette leggendo: 4 commenti per posizione, metà commenti all'articolo e metà risposte, estratti a caso (seed fisso). I **favorevoli** vengono tutti da prima del 2025, perché il loro peso nel 2003-2016 (49%) è il risultato più sorprendente; per **critici** e **neutri** un esempio prima e uno dal 2025 per tipo. Nella cella successiva, un esempio per ogni emozione. Testi tagliati a 500 caratteri (commento) e 200 (commento precedente); pochi esempi casuali servono a scoprire errori sistematici evidenti, non a stimare l'accuratezza.

In [ ]:
corpus_examples = comment_sentiment.merge(selected[["comment_id", "headline", "parent_text", "text"]], on="comment_id")
pre_2025 = corpus_examples["period"] != "2025-presente"


def example_table(rows: pd.DataFrame) -> "pd.io.formats.style.Styler":
    return (pd.DataFrame({
        "stance": rows["llm_stance"].map(stance_full),
        "emotion": rows["llm_emotion"],
        "type": rows["type"],
        "period": rows["period"].astype(str),
        "article headline": rows["headline"],
        "previous comment": rows["parent_text"].map(lambda s: cut(s, 200)),
        "comment": rows["text"].map(lambda s: cut(s, 500)),
        "reasoning (Haiku)": rows["llm_reasoning"],
    }).reset_index(drop=True).style
     .hide(axis="index")
     .set_properties(**{"text-align": "left", "vertical-align": "top", "white-space": "normal"})
     .set_properties(subset=["comment", "previous comment", "reasoning (Haiku)"], **{"min-width": "260px"})
     .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}]))


rows = []
for stance in ("fav", "crit", "neu"):
    for typ in ("top-level", "reply"):
        g = corpus_examples[(corpus_examples["llm_stance"] == stance) & (corpus_examples["type"] == typ)]
        if stance == "fav":
            rows.append(g[pre_2025].sample(2, random_state=SEED))
        else:
            rows += [g[pre_2025].sample(1, random_state=SEED), g[~pre_2025].sample(1, random_state=SEED)]
example_table(pd.concat(rows))

In [ ]:
example_table(pd.concat([corpus_examples[corpus_examples["llm_emotion"] == emo].sample(1, random_state=SEED)
                         for emo in COMMENT_EMOTION_LABELS]))